# Silver Transformation - dimcurrency

This notebook builds the `dimcurrency` dataset in the Silver layer.



## Run Shared Libraries



In [ ]:
%run ../Misc/sharedlibraries

In [ ]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimcurrency"

## Read Source Tables



In [ ]:
currencyDf = spark.table("bronze.currency")

In [ ]:
currencyDf.printSchema()

## Build Silver Dataset



In [ ]:
dimcurrencyDf = currencyDf.filter(currencyDf.RecordId.isNotNull()
    ).select(
       currencyDf.CurrencyId,
       F.trim(currencyDf.Code).alias("CurrencyCode"),
       F.when(currencyDf.LastProcessedChange_DateTime.isNull(), F.lit("1900-01-01")).otherwise(currencyDf.LastProcessedChange_DateTime).cast("timestamp").alias("LastProcessedChange_DateTime"),
       F.from_utc_timestamp(currencyDf.DataLakeModified_DateTime,'CST').alias("DataLakeModified_DateTime"),
       F.trim(currencyDf.Country).alias("Country"),
       F.trim(currencyDf.CurrencyName).alias("CurrencyName"),
       currencyDf.RecordId.alias("CurrencyRecordId")
    ).withColumn("UpdatedDateTime", F.lit(UpdatedDateTime)
    ).withColumn("CurrencyHashKey", F.xxhash64("CurrencyRecordId")
    )

display(dimcurrencyDf)

## Prepare Final DataFrame



In [ ]:
df_final = dimcurrencyDf

## Verify Source Schema



In [ ]:
saveDeltaTableToCatalog(df_final,"silver",Entity)